# 📊 Análisis del Mercado Laboral TI — Analista de Datos Chile 2025

**Autor:** Emilio Rubina Salinas  
**LinkedIn:** [linkedin.com/in/emilio-rubina-salinas-b1436a253](https://www.linkedin.com/in/emilio-rubina-salinas-b1436a253/)  
**Fuentes:** LinkedIn Jobs + Trabajando.cl  
**Período de recopilación:** Mayo–Junio 2026  

## Objetivo
Análisis exploratorio de 89 ofertas laborales del área de datos en Chile para identificar:
- Skills técnicos más demandados
- Distribución de seniority y modalidad de trabajo
- Sectores e industrias con mayor demanda
- Herramientas de BI prevalentes en el mercado
- Calidad metodológica de las plataformas (duplicados, ruido)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BLUE = '#1A56A0'
GREEN = '#3B6D11'
AMBER = '#BA7517'
RED = '#A32D2D'
GRAY = '#888780'
LIGHT = '#EBF3FC'

print('Librerías cargadas ✓')

## 1. Carga y exploración inicial de datos

In [ ]:
df = pd.read_csv('ofertas_ti_chile_clean.csv')

print(f'Total registros: {len(df)}')
print(f'Ofertas únicas: {df[~df["Es_Duplicado"]].shape[0]}')
print(f'Duplicados detectados: {df["Es_Duplicado"].sum()}')
print(f'\nColumnas: {list(df.columns)}')
df.head(3)

In [ ]:
# Resumen estadístico
print('=== RESUMEN DEL DATASET ===')
print(f"Plataformas: {df['Fuente'].value_counts().to_dict()}")
print(f"Industrias únicas: {df['Industria'].nunique()}")
print(f"Empresas únicas: {df['Empresa'].nunique()}")
print(f"Regiones: {df['Region'].nunique()}")
print(f"\nTop 5 empresas con más ofertas:")
print(df['Empresa'].value_counts().head(5).to_string())

## 2. Calidad de datos — Ruido y duplicados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Calidad del Dataset — Ruido y Duplicados', fontsize=14, fontweight='bold', color=BLUE)

# Duplicados por plataforma
dup_data = df.groupby('Fuente')['Es_Duplicado'].sum()
total_data = df.groupby('Fuente').size()
pct_dup = (dup_data / total_data * 100).round(1)

bars = axes[0].bar(pct_dup.index, pct_dup.values, color=[BLUE, AMBER], edgecolor='white', linewidth=1.5)
axes[0].set_title('% Duplicados por plataforma', fontweight='bold')
axes[0].set_ylabel('% de publicaciones duplicadas')
for bar, val in zip(bars, pct_dup.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f'{val}%', ha='center', fontweight='bold')

# Tipo de rol (ruido vs relevante)
tipo_counts = df['Tipo_Rol'].value_counts()
colors_tipo = [BLUE, '#3B8FDB', AMBER, GREEN, RED, GRAY]
wedges, texts, autotexts = axes[1].pie(tipo_counts.values, labels=tipo_counts.index, 
    autopct='%1.1f%%', colors=colors_tipo[:len(tipo_counts)],
    pctdistance=0.8, startangle=90)
for text in autotexts:
    text.set_fontsize(9)
axes[1].set_title('Distribución por tipo de rol', fontweight='bold')

plt.tight_layout()
plt.savefig('01_calidad_datos.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: ~57% de las ofertas son BI/Analytics puros. ~8% son ruido (roles no TI mal etiquetados).')

## 3. Skills más demandados

In [ ]:
skill_cols = [c for c in df.columns if c.startswith('skill_')]
skill_names = [c.replace('skill_','').replace('_',' ') for c in skill_cols]
skill_totals = df[skill_cols].sum().values

# Ordenar
sorted_idx = np.argsort(skill_totals)[::-1]
skill_names_sorted = [skill_names[i] for i in sorted_idx]
skill_vals_sorted = [skill_totals[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(12, 7))

colors = []
for v in skill_vals_sorted:
    if v >= 50: colors.append(BLUE)
    elif v >= 30: colors.append('#3B8FDB')
    elif v >= 15: colors.append(AMBER)
    else: colors.append(GRAY)

bars = ax.barh(skill_names_sorted, skill_vals_sorted, color=colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, skill_vals_sorted):
    pct = round(val / len(df) * 100)
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{val} ({pct}%)', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Número de ofertas', fontsize=11)
ax.set_title('🔧 Skills más demandados en el mercado TI Chile — Analista de Datos', 
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
ax.set_xlim(0, max(skill_vals_sorted) * 1.18)
ax.invert_yaxis()

patch_high = mpatches.Patch(color=BLUE, label='Alta demanda (≥50 ofertas)')
patch_mid = mpatches.Patch(color='#3B8FDB', label='Media (30-49)')
patch_low = mpatches.Patch(color=AMBER, label='Emergente (15-29)')
patch_niche = mpatches.Patch(color=GRAY, label='Nicho (<15)')
ax.legend(handles=[patch_high, patch_mid, patch_low, patch_niche], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('02_skills_demandados.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distribución de Seniority y Modalidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Seniority y Modalidad de Trabajo', fontsize=14, fontweight='bold', color=BLUE)

# Seniority
sen_order = ['Junior', 'Junior / Semi-Senior', 'Semi-Senior', 'Senior', 'No especificado']
sen_colors = [GREEN, '#7AB648', AMBER, RED, GRAY]
sen_counts = df['Seniority_norm'].value_counts().reindex(sen_order, fill_value=0)

bars1 = axes[0].bar(sen_counts.index, sen_counts.values, color=sen_colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución por Seniority', fontweight='bold')
axes[0].set_ylabel('Número de ofertas')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars1, sen_counts.values):
    pct = round(val / len(df) * 100)
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
                 f'{val}\n({pct}%)', ha='center', fontsize=9, fontweight='bold')

# Añadir línea de accesibilidad para juniors
junior_total = sen_counts['Junior'] + sen_counts['Junior / Semi-Senior']
axes[0].axhline(y=junior_total/2, color=GREEN, linestyle='--', alpha=0.4)
axes[0].text(4.4, junior_total/2 + 0.5, f'Accesibles\npara ti: {junior_total}', 
             color=GREEN, fontsize=8, ha='right')

# Modalidad
mod_colors = {'Híbrido': BLUE, 'Presencial': AMBER, 'Remoto': GREEN, 'No especificado': GRAY}
mod_counts = df['Modalidad_norm'].value_counts()
mod_c = [mod_colors.get(m, GRAY) for m in mod_counts.index]

wedges, texts, autotexts = axes[1].pie(mod_counts.values, labels=mod_counts.index,
    autopct='%1.1f%%', colors=mod_c, pctdistance=0.75,
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight('bold')
axes[1].set_title('Modalidad de trabajo', fontweight='bold')

plt.tight_layout()
plt.savefig('03_seniority_modalidad.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'💡 Insight: {junior_total} ofertas ({round(junior_total/len(df)*100)}%) accesibles para perfiles junior.')

## 5. Industrias y sectores

In [ ]:
ind_counts = df['Industria'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(11, 6))

cmap = plt.cm.Blues
norm = plt.Normalize(ind_counts.min(), ind_counts.max())
colors_ind = [cmap(norm(v)) for v in ind_counts.values]

bars = ax.barh(ind_counts.index[::-1], ind_counts.values[::-1], color=colors_ind[::-1], edgecolor='white')

for bar, val in zip(bars, ind_counts.values[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
            str(val), va='center', fontsize=10, fontweight='bold', color=BLUE)

ax.set_xlabel('Número de ofertas', fontsize=11)
ax.set_title('🏢 Sectores con mayor demanda de analistas de datos en Chile', 
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
ax.set_xlim(0, ind_counts.max() * 1.15)

plt.tight_layout()
plt.savefig('04_industrias.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Herramientas BI: Power BI vs Tableau vs Looker vs otros

In [ ]:
bi_tools = {
    'Power BI': df['skill_Power_BI'].sum(),
    'Tableau': df['skill_Tableau'].sum(),
    'Looker': df['skill_Looker'].sum(),
    'Excel (BI)': df['skill_Excel'].sum(),
}

fig, ax = plt.subplots(figsize=(9, 5))

bi_colors = [BLUE, AMBER, GREEN, '#666666']
bars = ax.bar(bi_tools.keys(), bi_tools.values(), color=bi_colors, edgecolor='white', linewidth=1.5, width=0.5)

for bar, val in zip(bars, bi_tools.values()):
    pct = round(val / len(df) * 100)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}\n{pct}%', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Número de ofertas', fontsize=11)
ax.set_title('📊 Herramientas de visualización — Prevalencia en el mercado chileno',
             fontsize=12, fontweight='bold', color=BLUE, pad=15)
ax.set_ylim(0, max(bi_tools.values()) * 1.2)

# Anotación certificación
ax.annotate('← Tu certificación\nPower BI Intermedio', 
            xy=(0, bi_tools['Power BI']), xytext=(0.8, bi_tools['Power BI'] * 0.7),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2),
            color=GREEN, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('05_herramientas_bi.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Salarios (ofertas con datos explícitos)

In [ ]:
salarios_df = df[~df['Salario'].isin(['', 'A convenir', 'No especificado', 'Acorde al mercado', 'Renta mercado', 'nan'])].copy()

print('Ofertas con salario explícito:')
print(salarios_df[['Cargo','Empresa','Salario','Seniority_norm']].to_string(index=False))
print(f'\n💡 Solo {len(salarios_df)} de {len(df)} ofertas ({round(len(salarios_df)/len(df)*100)}%) publicaron salario explícito.')
print('Esto confirma que la opacidad salarial es norma en el mercado TI chileno.')

## 8. Heatmap — Skills por tipo de rol

In [ ]:
skill_cols = [c for c in df.columns if c.startswith('skill_')]
skill_labels = [c.replace('skill_','').replace('_',' ') for c in skill_cols]

heatmap_data = df.groupby('Tipo_Rol')[skill_cols].mean() * 100
heatmap_data.columns = skill_labels

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(heatmap_data.values, cmap='Blues', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(skill_labels)))
ax.set_xticklabels(skill_labels, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=10)

for i in range(len(heatmap_data.index)):
    for j in range(len(skill_labels)):
        val = heatmap_data.values[i, j]
        if val > 0:
            ax.text(j, i, f'{val:.0f}%', ha='center', va='center', 
                    fontsize=7.5, color='white' if val > 55 else 'black', fontweight='bold')

plt.colorbar(im, ax=ax, label='% de ofertas que mencionan el skill')
ax.set_title('🔥 Skills por tipo de rol — Heatmap de demanda', 
             fontsize=13, fontweight='bold', color=BLUE, pad=15)
plt.tight_layout()
plt.savefig('06_heatmap_skills_rol.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusiones y recomendaciones

In [ ]:
print('='*60)
print('CONCLUSIONES DEL ANÁLISIS')
print('='*60)

total = len(df)
uniques = df[~df['Es_Duplicado']].shape[0]

conclusions = [
    f'1. SQL es el skill más demandado: {df["skill_SQL"].sum()}/{total} ofertas ({round(df["skill_SQL"].sum()/total*100)}%)',
    f'2. Power BI en {df["skill_Power_BI"].sum()} ofertas ({round(df["skill_Power_BI"].sum()/total*100)}%) — dominante en el mercado local',
    f'3. Python en {df["skill_Python"].sum()} ofertas — segunda herramienta más demandada después de SQL',
    f'4. Híbrido es la modalidad dominante: {(df["Modalidad_norm"]=="Híbrido").sum()} ofertas ({round((df["Modalidad_norm"]=="Híbrido").sum()/total*100)}%)',
    f'5. Solo el {round(len(salarios_df)/total*100)}% de las ofertas publica salario explícito — opacidad salarial alta',
    f'6. {round(df["Es_Duplicado"].sum()/total*100)}% de las publicaciones son duplicados (principalmente Trabajando.cl)',
    f'7. IA generativa (Gemini/ChatGPT) ya aparece como req. en {df["skill_IA_GenAI"].sum()} ofertas — tendencia emergente',
    f'8. Retail y Banca son los sectores con mayor volumen de ofertas',
    f'9. DAX aparece en {df["skill_DAX"].sum()} ofertas — es la siguiente certificación recomendada tras Power BI',
    f'10. {(df["Seniority_norm"].isin(["Junior","Junior / Semi-Senior"])).sum()} ofertas ({round((df["Seniority_norm"].isin(["Junior","Junior / Semi-Senior"])).sum()/total*100)}%) accesibles para perfiles junior',
]

for c in conclusions:
    print(f'  {c}')